# PneumoX-CL: Reproducibility Notebook

This notebook contains the full implementation used in the paper in a single Colab workflow.

Run all cells from top to bottom to reproduce the main experiments, including dataset preparation, sequential domain construction, baselines, and the proposed PneumoX-CL method.


## 1. Environment Setup

In [ ]:
!pip install -r requirements.txt

## 2. Imports and Data Path Configuration

In [ ]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import transforms
import medmnist
from medmnist import INFO
import matplotlib.pyplot as plt
import random
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from sklearn.manifold import TSNE

# Generalized dataset path for Colab / local execution.
# You can override this by setting the MEDMNIST_ROOT environment variable.
dataset_root = os.environ.get("MEDMNIST_ROOT", "/content/data/MedMNIST")
Path(dataset_root).mkdir(parents=True, exist_ok=True)
print(f"Dataset root: {dataset_root}")


## 3. Reproducibility Setup

In [ ]:
# === Setup ===
torch.manual_seed(0)
random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data_flag = 'pneumoniamnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

## 4. Dataset and Domain Construction

In [6]:
##########################################################
# === Domain Transforms ===
transform_base = transforms.ToTensor()

# === Domain Shift Transformations ===
class LowRadTransform:
    # Simulates low-dose X-ray by adding device noise (Poisson noise) and potentially reducing intensity.
    def __init__(self, scale_factor=0.5, noise_level=0.1):
        self.scale_factor = scale_factor
        self.noise_level = noise_level

    def __call__(self, img):
        # Ensure img is a tensor before applying operations
        if not isinstance(img, torch.Tensor):
             raise TypeError("Input to LowDoseTransform must be a PyTorch Tensor.")

        img = img * self.scale_factor
        # Add a small epsilon to prevent division by zero in Poisson noise if noise_level is very small or zero
        noise = torch.poisson(img / (self.noise_level + 1e-6)) * self.noise_level
        img = img + noise
        img = torch.clamp(img, 0, 1)  # Ensure values are within a valid range
        return img

class MobileTransform:
    # Simulates portable X-ray in ICU by applying a subtle change in brightness and potentially blurring.
    # These images can often have non-uniform illumination or artifacts due to the limitations of portable equipment and the patient's position.
    def __init__(self, gradient_strength=0.1, blur_radius=1):
        self.gradient_strength = gradient_strength
        self.blur_radius = blur_radius

    def __call__(self, img):
        # Add a subtle gradient
        height, width = img.shape[-2:]
        gradient = torch.linspace(0, self.gradient_strength, width).unsqueeze(0).repeat(height, 1)
        if img.ndim == 3:
            gradient = gradient.unsqueeze(0) # Add channel dimension if needed
        img = img + gradient
        img = torch.clamp(img, 0, 1)

        # Apply subtle blur (using a simple average filter for demonstration)
        if self.blur_radius > 0:
            avg_pool = nn.AvgPool2d(kernel_size=self.blur_radius*2+1, stride=1, padding=self.blur_radius)
            img = avg_pool(img.unsqueeze(0)).squeeze(0)

        return img

class DemographicShiftTransform:
    # Simulates anatomical variations by applying a small random translation or scaling.
    # Emulates different body sizes or anatomical presentations (e.g., pediatric or obese).
    def __init__(self, max_translation=2, max_scale_factor=0.1, output_size=(28, 28)):
        self.max_translation = max_translation
        self.max_scale_factor = max_scale_factor
        self.output_size = output_size

    def __call__(self, img):
        # Random translation
        dx = random.randint(-self.max_translation, self.max_translation)
        dy = random.randint(-self.max_translation, self.max_translation)
        img = torch.roll(img, shifts=(dy, dx), dims=(-2, -1))

        # Random scaling (simple resize for demonstration)
        scale_factor = 1.0 + random.uniform(-self.max_scale_factor, self.max_scale_factor)
        if scale_factor != 1.0:
            # Need to resize while handling potential channel dimension
            if img.ndim == 3:
                c, h, w = img.shape
                new_h, new_w = int(h * scale_factor), int(w * scale_factor)
                # Use transforms.functional.resize which handles tensor input
                img = transforms.functional.resize(img, (new_h, new_w))

        # Ensure the image is exactly the output_size using CenterCrop after potential scaling/translation
        img = transforms.functional.center_crop(img, self.output_size)

        return img

class HospitalShiftTransform:
    # Simulates differences between institutions by applying variations in contrast, brightness, or sharpness.
    # replicate the variations that exist between medical images acquired at different institutions (hospitals, clinics, etc.).
    # These differences can arise from different equipment, different imaging protocols, different post-processing.
    def __init__(self, brightness_factor=0.2, contrast_factor=0.2, sharpness_factor=0.2):
        self.brightness_factor = brightness_factor
        self.contrast_factor = contrast_factor
        self.sharpness_factor = sharpness_factor

    def __call__(self, img):
        # Apply random brightness, contrast, and sharpness adjustments
        img = transforms.functional.adjust_brightness(img, 1.0 + random.uniform(-self.brightness_factor, self.brightness_factor))
        img = transforms.functional.adjust_contrast(img, 1.0 + random.uniform(-self.contrast_factor, self.contrast_factor))
        # Sharpness adjustment is more complex, a simple approximation is adding/subtracting a blurred version
        if self.sharpness_factor > 0:
            blurred_img = transforms.functional.gaussian_blur(img, kernel_size=3)
            img = img + (img - blurred_img) * self.sharpness_factor

        img = torch.clamp(img, 0, 1)
        return img

domain_names = ["Standard", "LowRad", "Mobile", "Demographic", "Hospital"]
domain_transforms = [
    transforms.ToTensor(), # Base transform (just ToTensor)
    transforms.Compose([transforms.ToTensor(), LowRadTransform()]),
    transforms.Compose([transforms.ToTensor(), MobileTransform()]),
    transforms.Compose([transforms.ToTensor(), DemographicShiftTransform()]),
    transforms.Compose([transforms.ToTensor(), HospitalShiftTransform()])
]

## 5. Model Definitions

In [7]:
# === MODEL 1: MLP ===
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)   # Binary class
        )

    def forward(self, x):
        return self.layers(x)

# === MODEL 2: CNN ===
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# === MODEL 3: MobileNetV2 ===
class MobileNetV2(nn.Module):
    def __init__(self, num_classes=2):
        super(MobileNetV2, self).__init__()
        self.mobilenet = models.mobilenet_v2(pretrained=False)
        # Change input conv layer to accept 1-channel input
        self.mobilenet.features[0][0] = nn.Conv2d(
            1, 32, kernel_size=3, stride=2, padding=1, bias=False
        )
        in_features = self.mobilenet.classifier[1].in_features
        self.mobilenet.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        x = self.mobilenet(x)
        return x

import torchvision.models as models

# === MODEL 4: CNN_with_Feature ===
# used with Strata_Diversity buffer; CNN returns logits + features (for cosine calculation between images)
class CNN_with_Feature(nn.Module):
    def __init__(self):
        super(CNN_with_Feature, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        feat = self.relu(self.fc1(x))  # Use this for diversity
        x = self.dropout(feat)         # Only use in classification path
        logits = self.fc2(x)
        return logits, feat

class MobileNetV2_Feature(nn.Module):
    def __init__(self, num_classes=2):
        super(MobileNetV2_Feature, self).__init__()
        self.mobilenet = models.mobilenet_v2(pretrained=False)

        # Change input conv layer to accept 1 channel
        self.mobilenet.features[0][0] = nn.Conv2d(
            1, 32, kernel_size=3, stride=2, padding=1, bias=False
        )
        # Extract the feature dimension from classifier
        in_features = self.mobilenet.classifier[1].in_features

        # Save a copy of the original classifier if needed
        self.mobilenet.classifier_original = self.mobilenet.classifier

        # Replace classifier to match target classes
        self.mobilenet.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        # Get features from the backbone
        x = self.mobilenet.features(x)
        x = x.mean([2, 3])  # Global average pooling
        feat = x  # This will be used for diversity-aware sampling
        logits = self.mobilenet.classifier(x)
        return logits, feat

## 6. Replay Buffer Implementations

In [8]:
# === THREE Experience Replay Buffer ===

###############################################
# Buffer 1: Classic Reservoir buffer
class ReservoirBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.n_seen = 0

    # Add a batch of new experiences (images and their corresponding labels) into the replay buffer.
    def add_batch(self, images, labels):
        for x, y in zip(images, labels):
            self.n_seen += 1
            if len(self.buffer) < self.capacity:
                self.buffer.append((x.cpu(), y.cpu()))
            else:
                idx = random.randint(0, self.n_seen - 1)
                if idx < self.capacity:
                    self.buffer[idx] = (x.cpu(), y.cpu())

    # Retrieves a random batch of experiences from the buffer for use during training.
    def sample(self, batch_size):
        if len(self.buffer) == 0:
            return None, None
        samples = random.sample(self.buffer, min(batch_size, len(self.buffer)))
        imgs, labels = zip(*samples)
        return torch.stack(imgs), torch.tensor(labels)

    def print_label_distribution(self):
        if len(self.buffer) == 0:
            print("Buffer is empty.")
            return

        # Extract labels as Python ints
        labels = [int(y.item()) for _, y in self.buffer]

        counts = Counter(labels)
        for lbl in sorted(counts.keys()):
            print(f"Label {lbl}: {counts[lbl]} samples")

        print(f"Total samples in buffer: {len(self.buffer)}")

###############################################
# Buffer 2: Class-balanced Reservoir buffer
class CBRSBuffer:
    def __init__(self, capacity_per_class=5, num_classes=10):
        self.capacity_per_class = capacity_per_class
        self.num_classes = num_classes
        self.buffer = {cls: [] for cls in range(num_classes)}
        self.class_seen_count = {cls: 0 for cls in range(num_classes)}

    def add_batch(self, images, labels):
        for x, y in zip(images, labels):
            cls = y.item()
            self.class_seen_count[cls] += 1
            seen = self.class_seen_count[cls]
            #print(f"[CBRS] Seen class {cls}: {seen} samples")
            if len(self.buffer[cls]) < self.capacity_per_class:
                self.buffer[cls].append((x.cpu(), y.cpu()))
                #print(f"[CBRS] -> Added to class {cls} buffer (size now {len(self.buffer[cls])})")
            else:
                prob = self.capacity_per_class / seen
                rand_val = random.random()
                if rand_val < prob:
                    idx = random.randint(0, self.capacity_per_class - 1)
                    self.buffer[cls][idx] = (x.cpu(), y.cpu())
                    #print(f"[CBRS] -> Replaced entry in class {cls} buffer (rand={rand_val:.3f} < prob={prob:.3f})")
                #else:
                    #print(f"[CBRS] -> Skipped (rand={rand_val:.3f} >= prob={prob:.3f})")

    # Sampling is uniform across all stored examples.
    def sample(self, batch_size):
      all_samples = [sample for samples in self.buffer.values() for sample in samples]
      if not all_samples:
          return None, None

      selected = random.sample(all_samples, min(batch_size, len(all_samples)))
      x, y = zip(*selected)
      return torch.stack(x), torch.tensor(y)

from collections import defaultdict, Counter

#####################################################################
# Buffer 3: class Stratified_Diverse Buffer ===
class Strata_Diversity_Buffer:
    def __init__(self, capacity, num_classes):
        self.capacity = capacity
        self.num_classes = num_classes
        # Instead of a single buffer, we now have strata:
        # a dictionary where keys are class labels and values are lists to store samples for each class.
        self.strata = {i: [] for i in range(num_classes)}  # Strata for each class

    # The logic for adding samples is modified to work within the specific stratum for the sample's class.
    # Cosine similarity is applied within the stratum to replace the most similar existing sample if the stratum is full.
    def add_batch(self, images, labels, features):
        for x, y, feat in zip(images, labels, features):
            stratum = self.strata[y.item()]  # Get the stratum for this sample's class
            if len(stratum) < self.capacity // self.num_classes:
                stratum.append((x.cpu(), y.cpu(), feat.cpu()))  # Add directly if stratum has space
            else:
                buffer_features = torch.stack([f for _, _, f in stratum])

                # Cosine Similarity is used to calculate the similarity between the new sample and existing samples in the class stratum:
                similarities = torch.nn.functional.cosine_similarity(feat.unsqueeze(0), buffer_features)
                # The sample with the highest similarity is identified and replaced:
                max_similarity, replace_idx = torch.max(similarities, dim=0)

                #Above: Cosine, Below: Euclidean
                """
                # Compute Euclidean distances: ||feat - f||_2
                dists = torch.norm(buffer_features - feat, dim=1)
                # Find index of closest (most similar) sample
                replace_idx = torch.argmin(dists)
                """
                # # Replace most similar sample
                stratum[replace_idx.item()] = (x.cpu(), y.cpu(), feat.cpu())

    # Samples are drawn proportionally from each stratum to ensure class balance in the returned batch.
    # We check if the stratum is not empty before sampling to avoid errors.
    def sample(self, batch_size):
        samples = []
        for stratum in self.strata.values():
            # Sample proportionally from each stratum
            num_samples_from_stratum = batch_size // self.num_classes
            if stratum:  # Check if stratum is not empty
                samples.extend(random.sample(stratum, min(num_samples_from_stratum, len(stratum))))

        if not samples:
            return None, None

        imgs, labels, _ = zip(*samples)
        return torch.stack(imgs), torch.tensor(labels)

    def calculate_diversity(self):
        total_dissimilarity = 0
        total_pairs = 0

        for stratum in self.strata.values():
            if len(stratum) > 1:  # Need at least 2 samples to calculate diversity
                # Change this line to unpack only 3 elements
                features = torch.stack([f for _, _, f in stratum])
                similarities = torch.nn.functional.cosine_similarity(features[:, None], features, dim=2)
                dissimilarities = 1 - similarities
                # Sum dissimilarities for all pairs (excluding self-similarity)
                total_dissimilarity += dissimilarities.sum() - dissimilarities.trace()
                total_pairs += len(stratum) * (len(stratum) - 1)  # Number of unique pairs

        if total_pairs == 0:
            return 0  # Handle empty buffer or strata with single samples

        average_dissimilarity = total_dissimilarity / total_pairs
        return average_dissimilarity.item()  # Return as a single number

    # extract feature embeddings for t-SNE plot
    def get_all_features_and_labels(self):
        features = []
        labels = []
        for cls, stratum in self.strata.items():
            for _, y, feat in stratum:
                features.append(feat.numpy())
                labels.append(y.item())
        return np.array(features), np.array(labels)

    def print_label_distribution(self):
        total = 0
        for cls_id, stratum in self.strata.items():
            count = len(stratum)
            total += count
            print(f"Label {cls_id}: {count} samples")
        print(f"Total samples in buffer: {total}")

## 7. Training Utilities

In [9]:
# === Training Function ===
def train_with_er(model, loader, buffer):
    model.train()
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(num_epochs):
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for x, y in loop:
            x = x.to(device) # Keep original batch for buffer
            y = y.squeeze().long().to(device) # Keep original batch for buffer

            # Replay
            x_replay, y_replay = buffer.sample(int(replay_ratio * x.size(0)))
            if x_replay is not None:
                x = torch.cat([x, x_replay.to(device)])
                y = torch.cat([y, y_replay.to(device)])

            # Train
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()

            # Add to buffer
            buffer.add_batch(x[:batch_size].detach().cpu(), y[:batch_size].detach().cpu())

            # Update tqdm description with loss
            loop.set_postfix(loss=loss.item())

###################################################
# === Online Class Frequency Tracker + train_with_er_dynamic_weight ===
class ClassFrequencyTracker:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.counts = torch.zeros(num_classes)
        self.total_seen = 0

    # updating the internal count of each class as new data is processed.
    def update(self, labels):
        for label in labels:
            self.counts[label.item()] += 1
        self.total_seen += len(labels)

    # calculates and returns the weights for each class based on their frequencies
    def get_weights(self, normalize=True):
        epsilon = 1e-6
        freqs = self.counts.clone()
        freqs[freqs == 0] = epsilon
        inv_freqs = self.total_seen / freqs
        if normalize:
            inv_freqs = inv_freqs / inv_freqs.sum() * self.num_classes
        return inv_freqs

def train_with_er_dynamic_weight(model, loader, buffer):
    model.train()
    opt = optim.Adam(model.parameters(), lr=lr)
    #opt = optim.SGD(model.parameters(), lr=lr)

    tracker = ClassFrequencyTracker(num_classes=2)

    for epoch in range(num_epochs):
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        total_loss = 0
        for x, y_new in loop: # Renamed y to y_new for clarity
            x = x.to(device) # Keep original batch for buffer
            y_new = y_new.squeeze().long().to(device) # Keep original batch for buffer

            # Replay
            x_replay, y_replay = buffer.sample(int(replay_ratio * x.size(0)))

            if x_replay is not None:
                x_combined = torch.cat([x, x_replay.to(device)])
                y_combined = torch.cat([y_new, y_replay.to(device)])
            else:
                x_combined = x
                y_combined = y_new

            #################################################
            # update frequency count from current batch
            tracker.update(y_combined)
            weights = tracker.get_weights().to(device)
            loss_fn = nn.CrossEntropyLoss(weight=weights)
            #################################################

            # Train
            opt.zero_grad()
            loss = loss_fn(model(x_combined), y_combined) # Use the combined batch for loss calculation
            loss.backward()
            opt.step()

            # Add to buffer (add the new batch to the buffer)
            buffer.add_batch(x.detach().cpu(), y_new.detach().cpu())

            # Update tqdm description with loss
            #loop.set_postfix(loss=loss.item())
            # Accumulate Loss
            total_loss += loss.item()
        #print(f"Epoch {epoch+1}: Loss = {total_loss / len(loader):.4f}, Diversity = {diversity:.4f}")
        print(f"Epoch {epoch+1}: Loss = {total_loss / len(loader):.4f}")

        # Removed the print statements for accumulated class counts and weights
        #print(f"[Epoch {epoch+1}] Current weights: {weights.tolist()}")


# === for Joing-training & Fine-tuning method ===
def train_on_domain(model, train_loader):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch_idx, (x, y) in enumerate(loop):
            x, y = x.to(device), y.squeeze().long().to(device) # Fix: Squeeze the target tensor

            # Ensure y is at least 1-dimensional
            if y.ndim == 0:
                y = y.unsqueeze(0)

            # Add this check to skip empty batches after squeezing and unsqueezing
            if y.size(0) == 0:
                continue

            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            loop.set_postfix(batch=batch_idx+1, loss=loss.item())
        print(f"  Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

# === Evaluation ===
def evaluate(model, loader, name=""):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            #x, y = x.to(device), y.to(device)
            x, y = x.to(device), y.squeeze().long().to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    accuracy = 100 * correct / total
    print(f"{name:<12} Accuracy: {accuracy:.2f}%")
    return accuracy

###################################################################
# === Training for Strata_Diversity method ===
def train_with_feature(model, loader, buffer):
    model.train()
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(num_epochs):
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        total_loss = 0
        for x, y in loop:
            x = x.to(device) # Keep original batch for buffer
            y = y.squeeze().long().to(device) # Keep original batch for buffer

            ###################################################################3
            # Step 1. Forward pass to extract features
            logits, features = model(x)

            # Replay
            x_replay, y_replay = buffer.sample(int(replay_ratio * x.size(0)))
            if x_replay is not None:
                logits_replay, _ = model(x_replay.to(device))
                x = torch.cat([x, x_replay.to(device)])
                y = torch.cat([y, y_replay.to(device)])
                logits = torch.cat([logits, logits_replay])

            # Train
            opt.zero_grad()
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()

            #################################################################33
            # Step 2. Add to buffer with features
            buffer.add_batch(x[:batch_size].detach().cpu(), y[:batch_size].detach().cpu(), features[:batch_size].detach().cpu())

            # Accumulate Loss
            total_loss += loss.item()

        # log
        diversity = buffer.calculate_diversity()
        print(f"Epoch {epoch+1}: Loss = {total_loss / len(loader):.4f}, Diversity = {diversity:.4f}")

def train_with_er_dynamic_weight_feature(model, loader, buffer):
    model.train()
    opt = optim.Adam(model.parameters(), lr=lr)
    #opt = optim.SGD(model.parameters(), lr=lr)

    tracker = ClassFrequencyTracker(num_classes=2)

    for epoch in range(num_epochs):
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        total_loss = 0
        for x, y_new in loop: # Renamed y to y_new for clarity
            x = x.to(device) # Keep original batch for buffer
            y_new = y_new.squeeze().long().to(device) # Keep original batch for buffer

            ###################################################################3
            # Step 1. Forward pass to extract features
            logits, features = model(x)

            # Replay
            x_replay, y_replay = buffer.sample(int(replay_ratio * x.size(0)))
            if x_replay is not None:
                logits_replay, _ = model(x_replay.to(device))
                x_combined = torch.cat([x, x_replay.to(device)])
                y_combined = torch.cat([y_new, y_replay.to(device)])
                logits = torch.cat([logits, logits_replay])
            else:
                x_combined = x
                y_combined = y_new

            # update frequency count from current batch
            tracker.update(y_combined)
            weights = tracker.get_weights().to(device)
            loss_fn = nn.CrossEntropyLoss(weight=weights)

            # Train
            opt.zero_grad()
            loss = loss_fn(logits, y_combined) # Use the combined batch for loss calculation
            loss.backward()
            opt.step()

            # Add to buffer (add the new batch to the buffer)
            buffer.add_batch(x[:batch_size].detach().cpu(), y_new[:batch_size].detach().cpu(), features[:batch_size].detach().cpu())

            # Accumulate Loss
            total_loss += loss.item()

        # log
        diversity = buffer.calculate_diversity()
        print(f"Epoch {epoch+1}: Loss = {total_loss / len(loader):.4f}, Diversity = {diversity:.4f}")


# === Evaluation ===
def evaluate_feature(model, loader, name=""):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            #x, y = x.to(device), y.to(device)
            x, y = x.to(device), y.squeeze().long().to(device)
            #pred = model(x).argmax(1)
            logits, _ = model(x)      # CHANGE: two outputs
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    accuracy = 100 * correct / total
    print(f"{name:<12} Accuracy: {accuracy:.2f}%")
    return accuracy


## 8. Joint Training

In [ ]:
##################################################
# Joint-training
num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running Joint-training - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Initialize model for each run
    lr = 0.001
    num_epochs = 50
    batch_size = 32

    all_train_datasets = []   # merge all domains

    model = CNN().to(device)
    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("method: CNN + Joint-training")

    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} MedMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        # merge
        all_train_datasets.append(train_data)
        test_loader = DataLoader(test_data, batch_size=batch_size)
        seen_test_sets.append((name, test_loader))

        # merge all domain data
        joint_train_data = ConcatDataset(all_train_datasets)

    # Joint training
    train_loader = DataLoader(joint_train_data, batch_size=batch_size, shuffle=True)
    train_on_domain(model, train_loader)

    print("== Evaluation on all seen domains ==")
    for test_name, loader in seen_test_sets:
        acc = evaluate(model, loader, test_name)
        domain_accuracies[test_name].append(acc) # Store the accuracy for the domain

    # Forgetting and Accuracy Analysis for the current run
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })

# After the loop, calculate and print the average across all runs
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")

## 9. Fine-Tuning

In [ ]:
##################################################
# Fine-Tuning
num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running Fine-tuning - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Initialize model for each run
    lr = 0.001
    num_epochs = 50
    batch_size = 32

    model = CNN().to(device)
    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("method: CNN + Fine-tuning")

    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} PneumoniaMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size)

        # Fine-tune
        train_on_domain(model, train_loader)
        seen_test_sets.append((name, test_loader))

        print("== Evaluation on all seen domains ==")
        for test_name, loader in seen_test_sets:
            acc = evaluate(model, loader, test_name)
            domain_accuracies[test_name].append(acc)

    # Forgetting and Accuracy Analysis for the current run
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })

# After the loop, calculate and print the average across all runs
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")

## 10. Experience Replay (ER)

In [ ]:
##################################################
# ER

from collections import Counter
import numpy as np

num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running ER - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Initialize model for each run
    lr = 0.001
    num_epochs = 50
    batch_size = 32
    buffer_capacity = 2000
    replay_ratio = 1.0

    model = CNN().to(device)
    buffer = ReservoirBuffer(buffer_capacity)
    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("replay_ratio:", replay_ratio)
    print("buffer_capacity:", buffer_capacity)
    print("method: CNN + ER")

    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} PneumoniaMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size)

        # ER
        train_with_er(model, train_loader, buffer)
        seen_test_sets.append((name, test_loader))
        buffer.print_label_distribution()
        print("== Evaluation on all seen domains ==")
        for test_name, loader in seen_test_sets:
            acc = evaluate(model, loader, test_name)
            domain_accuracies[test_name].append(acc)

    # print the label distribution in buffer
    buffer.print_label_distribution()

    # Forgetting and Accuracy Analysis for the current run
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })

# After the loop, calculate and print the average across all runs
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")

## 11. Class-Balanced Reservoir Sampling (CBRS)

In [ ]:
##################################################
# CBRS
num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running CBRS - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Initialize model for each run
    lr = 0.001
    num_epochs = 50
    batch_size = 32
    buffer_capacity = 2000
    replay_ratio = 1.0

    model = CNN().to(device)

    buffer = CBRSBuffer(capacity_per_class=1000, num_classes=2)  # CBRS buffer
    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("replay_ratio:", replay_ratio)
    print("buffer_capacity:", buffer_capacity)
    print("method: Tiny_32_CNN + CBRS")

    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} PneumoniaMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size)

        train_with_er(model, train_loader, buffer)
        seen_test_sets.append((name, test_loader))

        print("== Evaluation on all seen domains ==")
        for test_name, loader in seen_test_sets:
            acc = evaluate(model, loader, test_name)
            domain_accuracies[test_name].append(acc)

    # Forgetting and Accuracy Analysis for the current run
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })

# After the loop, calculate and print the average across all runs
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")

## 12. Proposed PneumoX-CL

In [ ]:
# Proposed PneumoX-CL
# ER + Strata_Diversity buffer + WL
num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running CNN + Strata_Diverse + WL - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Initialize model for each run
    lr = 0.001
    num_epochs = 50
    batch_size = 32
    buffer_capacity = 2000
    replay_ratio = 1.0

    model = CNN_with_Feature().to(device)
    buffer = Strata_Diversity_Buffer(capacity=2000, num_classes=2)  # Adjust capacity and num_classes

    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("replay_ratio:", replay_ratio)
    print("buffer_capacity:", buffer_capacity)
    print("method: CNN + Strata_Diversity buffer + Weighted Loss")

    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} MedMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size)

        ## Weighted Loss
        train_with_er_dynamic_weight_feature(model, train_loader, buffer)
        seen_test_sets.append((name, test_loader))

        print("== Evaluation on all seen domains ==")
        for test_name, loader in seen_test_sets:
            acc = evaluate_feature(model, loader, test_name)
            domain_accuracies[test_name].append(acc)

    buffer.print_label_distribution()

    # Forgetting and Accuracy Analysis for the current run
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })


# After the loop, calculate and print the average across all runs
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")